# Tutorial 3: metafspm and VirtualPlantLab, one plant

Two platforms work on the same root system, each in its own language and process, exchanging their values at each
step through encoding3d:

- **metafspm** (Python) grows the roots: each segment's tip elongates as fast as its sugar allows (a **structural**
  model: it creates the segments);
- **VirtualPlantLab** (Julia) is the carbon model: it splits each segment into two tissue layers, grown by a
  PlantGraphs rule, and computes their sugar from the segment's mass (a **functional** model, which also adds
  entities of its own, at a finer scale).

Both models are toys, written for the tutorial: they show the coupling, not plant science.

A **coordinator** (encoding3d's, in Python) runs the session: it keeps the shared plant, passes each model the values
it needs (converting units, copying a segment's mass to its layers, averaging the layers' sugar back to the segment),
and tells each model when to run. This notebook starts the coordinator, launches VirtualPlantLab's model
(`carbon.jl`), and runs metafspm's.

In [ ]:
import os
import shutil
import threading
from dataclasses import dataclass

import numpy as np
import pandas as pd

from openalea import encoding3d
from openalea.encoding3d.coupling import Coordinator, Link
from openalea.metafspm.coupling.component import StructuralComponent, input_variable, parameter, state_variable
from openalea.metafspm.data_structure.configs import PropsConfig, ScalesConfig as scales
from openalea.metafspm.io.coupling import RemoteComponent
from openalea.metafspm.scene.scene import Scene, SceneRecorder
from openalea.metafspm.solve.decorator import actual, segmentation

OUTPUTS = "outputs"
JULIA = os.environ.get("JULIA", "julia")             # the Julia that runs VirtualPlantLab's model
STEPS = 8

## 1. metafspm's root growth

As in tutorial 2a, a component grows one root axis, but its tip now elongates in proportion to its sugar,
`C_hexose_root`, an **input**: metafspm does not compute it, VirtualPlantLab will. Each segment's structural mass is
proportional to its length.

In [ ]:
SEGMENT = 0.01                                       # m
DENSITY = 0.1                                        # g of structure per m of root
UNIT = dict(unit_comment="", value_comment="", references="", DOI=[], min_value=None, max_value=None)


def segment_variable(unit, description, default=0., kind="descriptor"):
    return state_variable(unit=unit, description=description, default=default, scale=scales.SubOrgan, kind=kind, **UNIT)


@dataclass
class SugarLimitedRoot(StructuralComponent):
    """A root axis whose tip elongates with its sugar, cut every 1 cm (a toy)."""
    C_hexose_root: float = input_variable(unit="mol.g-1", description="sugar concentration", default=1.,
                                          scale=scales.SubOrgan, kind="massic_concentration", **UNIT)
    length: float = segment_variable("m", "the segment's length", kind="extensive")
    struct_mass: float = segment_variable("g", "the segment's structural mass", kind="extensive")
    radius: float = segment_variable("m", "the segment's radius", default=0.0005, kind="intensive")
    is_apex: float = segment_variable("adim", "1 for the tip")
    x1: float = segment_variable("m", "proximal end")
    y1: float = segment_variable("m", "proximal end")
    z1: float = segment_variable("m", "proximal end")
    x2: float = segment_variable("m", "distal end")
    y2: float = segment_variable("m", "distal end")
    z2: float = segment_variable("m", "distal end")
    elongation_rate: float = parameter(unit="m.s-1.g.mol-1", description="elongation per unit of sugar",
                                       default=2e-7, **UNIT)

    @classmethod
    def initiate_plant(cls, g, plant, scenario):
        s = g.scales
        parent = plant
        for scale, label in ((s.Axis, g.labels.Axis.Root), (s.GrowthUnit, g.labels.GrowthUnit.Root),
                             (s.Phytomer, g.labels.Phytomer.Root), (s.Organ, g.labels.Organ.RootInternode)):
            parent = g.add_component(parent, **PropsConfig(scale=scale, edge_type="/", label=label))
        tip = g.add_component(parent, **PropsConfig(scale=s.SubOrgan, edge_type="/", label=g.labels.SubOrgan.RootSegment))
        x, y, z = (g.property(name)[plant] for name in ("x", "y", "z"))
        for name, value in (("length", 0.005), ("struct_mass", DENSITY * 0.005), ("radius", 0.0005), ("is_apex", 1.),
                            ("x1", x), ("y1", y), ("z1", z), ("x2", x), ("y2", y), ("z2", z - 0.005)):
            g.property(name)[tip] = value

    def _tips(self):
        return [v for v, flag in self.mtg.property("is_apex").items() if flag]

    @actual
    def _elongation(self):
        g = self.mtg
        length, mass, z2, sugar = (g.property(name) for name in ("length", "struct_mass", "z2", "C_hexose_root"))
        rate = dict(zip(self.data_structure.entity_ids("node").tolist(),
                        self.parameter_values("elongation_rate").tolist()))
        for tip in self._tips():
            growth = rate[tip] * sugar[tip] * self.dt
            length[tip] += growth
            mass[tip] = DENSITY * length[tip]
            z2[tip] -= growth

    @segmentation
    def _segmentation(self):
        g = self.mtg
        props = {name: g.property(name) for name in ("length", "struct_mass", "is_apex", "radius", "C_hexose_root",
                                                     "x1", "y1", "z1", "x2", "y2", "z2")}
        for tip in self._tips():
            if props["length"][tip] <= SEGMENT:
                continue
            rest = props["length"][tip] - SEGMENT
            props["length"][tip], props["struct_mass"][tip], props["is_apex"][tip] = SEGMENT, DENSITY * SEGMENT, 0.
            props["z2"][tip] = props["z1"][tip] - SEGMENT
            x, y, z = props["x2"][tip], props["y2"][tip], props["z2"][tip]
            g.add_child(tip, **PropsConfig(scale=g.scales.SubOrgan, edge_type="<", label=g.labels.SubOrgan.RootSegment,
                                           length=rest, struct_mass=DENSITY * rest, is_apex=1.,
                                           radius=props["radius"][tip], C_hexose_root=props["C_hexose_root"][tip],
                                           x1=x, y1=y, z1=z, x2=x, y2=y, z2=z - rest))

## 2. VirtualPlantLab's model, seen from metafspm

Inside metafspm, the other platform's model is a **RemoteComponent**: a component that computes nothing. Its state
variables are what the remote model gives (`C_hexose_root`), its inputs what metafspm sends it (`struct_mass`). It
also says what metafspm does with the plant's scales: metafspm **owns** the plant down to the segments (it creates
them); it **follows** the layers VirtualPlantLab adds (they are created in metafspm's graph too). And it **links**
metafspm's names to the other model's: the segment's sugar is the mean of its layers' `hexose`.

Calling it at the end of each step sends metafspm's outputs and brings back the next values.

In [ ]:
@dataclass
class VirtualPlantLabCarbon(RemoteComponent):
    """VirtualPlantLab's carbon model (carbon.jl), as a metafspm component."""
    participant = "metafspm"
    rights = {"Plant": "owns", "Axis": "owns", "GrowthUnit": "owns", "Phytomer": "owns", "Organ": "owns",
              "SubOrgan": "owns", "Layer": "follows"}
    links = [Link("metafspm", "C_hexose_root", "carbon", "hexose", aggregation="mean")]
    C_hexose_root: float = state_variable(unit="mol.g-1", description="sugar concentration", default=1.,
                                          scale=scales.SubOrgan, kind="massic_concentration", **UNIT)
    struct_mass: float = input_variable(unit="g", description="structural mass", default=0., scale=scales.SubOrgan,
                                        kind="extensive", **UNIT)


class CoupledRoot:
    """The plant model: metafspm's growth, then the exchange with VirtualPlantLab."""
    initiators = (SugarLimitedRoot,)

    def __init__(self, data_structure, time_step):
        self.growth = SugarLimitedRoot(data_structure=data_structure)
        self.carbon = VirtualPlantLabCarbon(data_structure=data_structure)
        self.components = [self.growth, self.carbon]
        self.carbon.start()                            # joins the session: the initial state exchanged

    def run(self):
        self.growth()
        self.carbon()

## 3. The session

The coordinator gets the session's settings: how many steps, how long each is, the links it applies itself (the
layers need their segment's structural mass: `copy` it down), and a folder where it **records** the whole shared
plant at each step, as an encoding3d sequence. Then it launches VirtualPlantLab's model, which joins, and waits for
metafspm, which joins when its scene is built.

The participants talk over ZeroMQ (in memory). `Coordinator("files:outputs/messages", ...)` would make them
exchange files in a folder instead, with no other change.

In [ ]:
for folder in ("record", "metafspm"):
    shutil.rmtree(os.path.join(OUTPUTS, folder), ignore_errors=True)
coordinator = Coordinator("tcp://127.0.0.1:*", steps=STEPS, interval=6., time_unit="h", expect=["metafspm"],
                          links=[Link("carbon", "struct_mass", "metafspm", "struct_mass", aggregation="copy")],
                          record=os.path.join(OUTPUTS, "record"), timeout=600)
coordinator.launch("carbon", [JULIA, "--project=.", "carbon.jl"])
session = threading.Thread(target=coordinator.run)
session.start()
VirtualPlantLabCarbon.endpoint = coordinator.endpoint
print("coordinator at", coordinator.endpoint)

metafspm's scene: one plant, six-hour steps, its own frames recorded too. Each `scene.run()` is one step of the
session.

In [ ]:
planting = pd.DataFrame([dict(plant="root", model=CoupledRoot, x=0., y=0., z=0., rotation=0., scenario={})])
planting.attrs.update(xrange=0.1, yrange=0.1)
scene = Scene(planting=planting, time_step=6 * 3600.,
              recorder=SceneRecorder(os.path.join(OUTPUTS, "metafspm"), summaries=False, frame_period=1))
for _ in range(STEPS):
    scene.run()
session.join()                                       # the coordinator stops the session after the last step
print("the session has ended")

## 4. What happened

The coordinator's record holds the shared plant at each step: metafspm's segments with their mass, and
VirtualPlantLab's layers with their sugar.

In [ ]:
rows = []
for frame in encoding3d.read_sequence(os.path.join(OUTPUTS, "record", "scene.fspm.json")):
    plant = frame.data_structure("plant")
    segments, layers = plant.entity_set("SubOrgan"), plant.entity_set("Layer")
    rows.append(dict(time_h=frame.time, segments=segments.count, layers=layers.count,
                     total_mass_mg=1000 * np.nansum(segments.columns["struct_mass"]),
                     mean_hexose=np.nanmean(layers.columns["hexose"]) if layers.count else np.nan))
pd.DataFrame(rows)

The sugar of a layer falls as its segment gets heavier, so the roots grow more slowly; new segments start with the
most sugar (their layers are new: their mass reaches VirtualPlantLab one step later, the coupling being explicit).
metafspm counts the mass in grams and VirtualPlantLab in milligrams: the coordinator converts, as it declares. metafspm's own frames
(`outputs/metafspm/frames`) hold the roots with their geometry and sugar: open them in ParaView with encoding3d's
plugin and colour the roots by `C_hexose_root`.

## Try it yourself

- Exchange through a folder: `Coordinator("files:outputs/messages", ...)`. The messages are deleted once read; add
  `keep_messages=True` to keep them and look at them.
- Change VirtualPlantLab's sugar rule in `carbon.jl`, and see metafspm's roots grow faster or slower.
- Run two plants in the scene: VirtualPlantLab's model splits the segments of both.